### 01. Import Depedencies

In [35]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,OneHotEncoder,OrdinalEncoder,LabelEncoder
)

### 02. Load Dataset

In [36]:
df = pd.read_csv("data/processed/binning_applied.csv")
df.head(5)

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bins
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,Newer
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,Medium
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Newer
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,Medium
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Newer


### 03. Data Prerocessing

In [37]:
ordinal_features = ["tenure_bins"]
nominal_features = ["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","Contract","PaperlessBilling","PaymentMethod"]
numerical_features = ["MonthlyCharges","TotalCharges"]
remainder_features = ["SeniorCitizen"] 
target_feature = "Churn"

### 3.1 Data Splitting

In [38]:
X = df.drop(columns=[target_feature])
Y = df[target_feature]

In [39]:
X_train,X_test,Y_train,Y_test = train_test_split(
    X,Y,
    test_size = 0.2,
    stratify = Y,
    random_state = 42
)

### 3.2 scaling and Encoding

In [40]:
numerical_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(strategy='median')),
                                    ('scaler', StandardScaler())
                                ]
                                )

norminal_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(
                                                            strategy='constant',
                                                            fill_value='missing'
                                                            )),
                                    ('encoder', OneHotEncoder())
                                ]
                                )

ordinal_transformer = Pipeline(
                                steps=[
                                    ('imputer', SimpleImputer(
                                                            strategy='constant',
                                                            fill_value='missing'
                                                            )),
                                    ('encoder', OrdinalEncoder())
                                ]
                                )

preprocessor = ColumnTransformer(
    transformers=[
        ('num',numerical_transformer,numerical_features),
        ('nom',norminal_transformer,nominal_features),
        ('ord',ordinal_transformer,ordinal_features)
    ],
    remainder='passthrough'
)

nominal_features_name=[]
for feature in nominal_features:
    unique_values=df[feature].unique()
    nominal_features_name.extend([f"{feature}_{val}" for val in unique_values])

In [41]:
X_train  = pd.DataFrame(
    preprocessor.fit_transform(X_train,Y_train),
    columns = numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [42]:
X_test = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=numerical_features+nominal_features_name+ordinal_features+remainder_features
)

In [43]:
le  = LabelEncoder()
Y_train  = pd.DataFrame(le.fit_transform(Y_train),columns=['Churn'])
Y_test  = pd.DataFrame(le.transform(Y_test),columns=['Churn'])

In [47]:
print(Y_train['Churn'].value_counts()/len(Y_train)*100)

Churn
0    73.422222
1    26.577778
Name: count, dtype: float64


### 4. Saving Files

In [48]:
X_train.to_csv("artifacts/X_train.csv",index=False)
Y_train.to_csv("artifacts/y_train.csv",index=False)
X_test.to_csv("artifacts/X_test.csv",index=False)
Y_test.to_csv("artifacts/y_test.csv",index=False)